# 01 — ToN-IoT Dataset Analysis

**Objective:** Review every record, feature column and label.

**Input:** Original ToN-IoT network CSV (read-only).

**Output:** Dataset summary, one graph and a JSON report.

**Labels:** `label` = Normal (0) / Attack (1); `type` = attack category. Labels are not model inputs.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path(r'D:\J26-CS-336-Network-Flow-V2\network_flow_v2')
DATA = ROOT / 'data/raw/ton_iot/ton_iot_train_test_network.csv'
AUDITS = ROOT / 'reports/audits'
FIGURES = ROOT / 'reports/figures'
REPORT = AUDITS / 'ton_iot_v3_dataset_analysis.json'
FIGURE = FIGURES / 'ton_iot_v3_class_balance.png'

LABELS = ['label', 'type']
# Candidate network-flow columns; Notebook 02 will justify the final selection.
CANDIDATES = [
    'proto', 'service', 'duration', 'src_bytes', 'dst_bytes',
    'conn_state', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes'
]
MISSING = {'', '-', '?', 'na', 'n/a', 'nan', 'none', 'null', 'missing'}
CHUNK_SIZE = 30000  # Memory limit per read, NOT a row limit.
EXPECTED_ROWS = 211043  # Count for this downloaded CSV, not the complete ToN-IoT release.

if not DATA.is_file():
    raise FileNotFoundError(f'Original dataset not found: {DATA}')
AUDITS.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
print('Using:', DATA)
print('All records will be processed; no sampling or filtering.')

## 1. Dataset columns

Confirm the available features and labels.

In [ ]:
columns = pd.read_csv(DATA, nrows=0).columns.tolist()
if len(columns) != len(set(columns)):
    raise ValueError('Duplicate CSV column names')
required = CANDIDATES + LABELS
missing_cols = [c for c in required if c not in columns]
if missing_cols:
    raise ValueError(f'Missing required columns: {missing_cols}')
print(f'Columns: {len(columns)} | Candidate flow features: {len(CANDIDATES)}')
display(pd.DataFrame({'candidate_feature': CANDIDATES}))

## 2. All rows and labels

Count every record and check the Normal/Attack target.

In [ ]:
counts = Counter()
types = Counter()
placeholders = Counter()
row_count = 0
label_type_mismatches = 0

for chunk in pd.read_csv(
    DATA, dtype=str, keep_default_na=False, chunksize=CHUNK_SIZE, low_memory=False,
    usecols=required
):
    row_count += len(chunk)
    y = chunk['label'].str.strip()
    t = chunk['type'].str.strip()
    counts.update(y.value_counts().to_dict())
    types.update(t.value_counts().to_dict())

    # A Normal row should have type 'normal' and an Attack row should not.
    is_normal_type = t.str.casefold().eq('normal')
    label_type_mismatches += int(((y.eq('0')) != is_normal_type).sum())

    for col in CANDIDATES:
        s = chunk[col].astype(str).str.strip().str.casefold()
        placeholders[col] += int(s.isin(MISSING).sum())

if set(counts) != {'0', '1'}:
    raise ValueError(f'Unexpected binary labels: {dict(counts)}')
if row_count != EXPECTED_ROWS:
    raise ValueError(f'Expected {EXPECTED_ROWS:,} rows for this CSV; found {row_count:,}. Review the source.')
if sum(counts.values()) != row_count:
    raise ValueError('Label totals do not equal dataset rows.')
if label_type_mismatches:
    raise ValueError(f'Binary/attack-type label mismatches: {label_type_mismatches:,}')

balance = pd.DataFrame([
    {'class': 'Normal', 'records': counts['0'], 'percentage': 100 * counts['0'] / row_count},
    {'class': 'Attack', 'records': counts['1'], 'percentage': 100 * counts['1'] / row_count}
])
print(f'Processed every row: {row_count:,}/{EXPECTED_ROWS:,} (100%). No rows removed.')
display(balance.round(2))
print('Attack types (including normal):', len(types))
display(pd.DataFrame([
    {'type': key, 'records': value} for key, value in types.most_common()
]))

## 3. Feature-quality preview

Count common missing or placeholder values. The detailed checks belong in Notebook 02.

In [ ]:
quality = pd.DataFrame([
    {'feature': feature, 'missing_or_placeholder': placeholders[feature],
     'percentage': round(100 * placeholders[feature] / row_count, 2)}
    for feature in CANDIDATES
]).sort_values('missing_or_placeholder', ascending=False)
display(quality)
print('Candidate columns are not yet the final training feature set.')

## 4. Graph and report

Save a class-balance graph and reproducible dataset summary.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(balance['class'], balance['records'])
for bar, row in zip(bars, balance.to_dict('records')):
    ax.annotate(f"{row['records']:,} ({row['percentage']:.2f}%)",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                xytext=(0, 5), textcoords='offset points', ha='center')
ax.set_ylim(0, balance['records'].max() * 1.20)
ax.set_title('ToN-IoT — Normal vs Attack')
ax.set_ylabel('Number of records')
ax.grid(axis='y', alpha=0.2)
fig.tight_layout()
fig.savefig(FIGURE, dpi=180, bbox_inches='tight')
plt.show()
plt.close(fig)

sha = hashlib.sha256()
with DATA.open('rb') as file:
    for block in iter(lambda: file.read(8 * 1024 * 1024), b''):
        sha.update(block)
report = {
    'dataset': 'ToN-IoT network CSV',
    'source_file': DATA.name,
    'source_sha256': sha.hexdigest(),
    'rows': row_count,
    'columns': len(columns),
    'column_names': columns,
    'candidate_flow_features': CANDIDATES,
    'target': 'label',
    'attack_type_label': 'type',
    'label_counts': dict(counts),
    'type_counts': dict(types),
    'candidate_feature_placeholder_counts': dict(placeholders),
    'label_type_mismatch_rows': label_type_mismatches,
    'all_rows_included': True,
    'note': 'Read-only audit, not a training run. No feature selection, splitting or cross-dataset transfer.'
}
REPORT.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Saved:', REPORT)
print('Saved:', FIGURE)
print('Next: 02_feature_selection.ipynb')